# Transcription Integration

Speech-to-text via Whisper or YouTube captions.

The transcription endpoint supports a `use_youtube_captions` flag:

- **`use_youtube_captions=True`** (default): YouTube captions are preferred when available, skipping Whisper entirely. This is faster since no GPU inference is needed. If YouTube captions are not available, Whisper runs as a fallback.
- **`use_youtube_captions=False`**: Whisper STT always runs on the video's audio track, regardless of whether YouTube captions exist. This produces more accurate timestamps and can provide word-level detail.

The transcription result is cached as JSON in `pipeline_data/api/transcriptions/whisper/`. Subsequent calls with `use_youtube_captions=True` return the cached result with `skipped=True`.

## Setup

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient

fw = FWClient()
print(f"Project root: {PROJECT_ROOT}")
print(f"Client: {fw}")

# Pick the first video from the registry
videos = fw.videos()
video_id = videos[0]["id"]
print(f"Using video: {videos[0]['title']} ({video_id})")

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-transcription")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /home/patrick-durzynski/foreign-whispers
Client: FWClient('http://localhost:8080')


ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/videos (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8080): Failed to establish a new connection: [Errno 111] Connection refused"))

## Run Transcription

The transcription endpoint has two modes:

1. **YouTube captions mode** (`use_youtube_captions=True`): Downloads and converts YouTube's auto-generated or manual captions into Whisper-compatible segment format. Fast, no GPU needed.
2. **Whisper STT mode** (`use_youtube_captions=False`): Runs the Whisper model on the video's audio track. Slower but produces higher-quality timestamps.

The `FWClient.transcribe()` method uses the default (YouTube captions). To force Whisper, we call the API directly with the query parameter.

In [3]:
# Transcribe using YouTube captions (default)
with logfire.span("transcribe.youtube_captions", video_id=video_id):
    result_yt = fw.transcribe(video_id)

print(f"Language: {result_yt['language']}")
print(f"Segment count: {len(result_yt['segments'])}")
print(f"Skipped (cached): {result_yt.get('skipped', False)}")
print()
print("First 3 segments:")
for seg in result_yt["segments"][:3]:
    duration = seg["end"] - seg["start"]
    print(f"  [{seg['start']:.2f}s - {seg['end']:.2f}s] ({duration:.2f}s) {seg['text']}")

NameError: name 'logfire' is not defined

## Force Whisper STT

You may want to force Whisper over YouTube captions when you need:

- **More accurate timestamps**: Whisper uses acoustic features to precisely locate speech boundaries, while YouTube captions often have coarser timing.
- **Word-level detail**: Whisper can produce word-level timestamps, useful for precise alignment in the TTS stage.
- **Consistency**: YouTube captions vary in quality across videos; Whisper provides a uniform baseline.

In [4]:
import requests

# Force Whisper STT by passing use_youtube_captions=False
# Note: this will take longer as Whisper runs on GPU
with logfire.span("transcribe.whisper_stt", video_id=video_id):
    resp = requests.post(
        f"{fw.base_url}/api/transcribe/{video_id}",
        params={"use_youtube_captions": False},
    )
    resp.raise_for_status()
    result_whisper = resp.json()

print(f"Language: {result_whisper['language']}")
print(f"Segment count: {len(result_whisper['segments'])}")
print(f"Skipped (cached): {result_whisper.get('skipped', False)}")
print()
print("First 3 segments:")
for seg in result_whisper["segments"][:3]:
    duration = seg["end"] - seg["start"]
    print(f"  [{seg['start']:.2f}s - {seg['end']:.2f}s] ({duration:.2f}s) {seg['text']}")

NameError: name 'logfire' is not defined

## Compare YouTube vs Whisper Segments

When both transcriptions are available, we can compare their segment characteristics:
- Segment count differences (YouTube captions often have fewer, longer segments)
- Duration distribution (Whisper tends to produce more uniform segment lengths)

In [5]:
import json
import matplotlib.pyplot as plt

# Load both transcription JSONs from disk if available
transcriptions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "transcriptions" / "whisper"
transcription_files = list(transcriptions_dir.glob("*.json"))

if not transcription_files:
    print("No transcription files found on disk. Run both transcription modes first.")
else:
    # Use the API results we already have
    yt_segments = result_yt["segments"]
    whisper_segments = result_whisper["segments"]

    yt_durations = [s["end"] - s["start"] for s in yt_segments]
    whisper_durations = [s["end"] - s["start"] for s in whisper_segments]

    print(f"YouTube captions: {len(yt_segments)} segments, "
          f"avg duration {sum(yt_durations)/len(yt_durations):.2f}s")
    print(f"Whisper STT:      {len(whisper_segments)} segments, "
          f"avg duration {sum(whisper_durations)/len(whisper_durations):.2f}s")

    # Plot segment duration distributions side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    axes[0].hist(yt_durations, bins=30, color="steelblue", edgecolor="white")
    axes[0].set_title("YouTube Captions")
    axes[0].set_xlabel("Segment duration (s)")
    axes[0].set_ylabel("Count")

    axes[1].hist(whisper_durations, bins=30, color="coral", edgecolor="white")
    axes[1].set_title("Whisper STT")
    axes[1].set_xlabel("Segment duration (s)")

    fig.suptitle("Segment Duration Distribution")
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "segment_duration_comparison.png", dpi=150)
    plt.show()
    print(f"Saved to {IMAGES_DIR / 'segment_duration_comparison.png'}")

NameError: name 'result_yt' is not defined

## Inspect Transcription JSON Structure

Each transcription is stored as a JSON file with the following top-level keys:
- `language`: detected source language code (e.g. `"en"`)
- `text`: full concatenated transcript text
- `segments`: list of segment objects with timing and text

In [6]:
import json

# Load one transcription JSON and inspect its structure
transcriptions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "transcriptions" / "whisper"
transcription_files = list(transcriptions_dir.glob("*.json"))

if transcription_files:
    data = json.loads(transcription_files[0].read_text())
    print(f"File: {transcription_files[0].name}")
    print(f"Top-level keys: {list(data.keys())}")
    print(f"Language: {data.get('language')}")
    print(f"Total segments: {len(data.get('segments', []))}")
    print()
    print("First 2 segments:")
    print(json.dumps(data["segments"][:2], indent=2))
else:
    print("No transcription files found on disk. Run the transcription step first.")

File: Strait of Hormuz disruption threatens to shake global economy.json
Top-level keys: ['language', 'text', 'segments']
Language: en
Total segments: 170

First 2 segments:
[
  {
    "id": 0,
    "start": 2.32,
    "end": 6.119999999999999,
    "text": "60 Minutes overtime."
  },
  {
    "id": 1,
    "start": 6.48,
    "end": 10.24,
    "text": "What's the worst case scenario that"
  }
]


## Summary

The transcription stage produces a JSON file per video in:

```
pipeline_data/api/transcriptions/whisper/<title>.json
```

Each segment in the JSON has the format:

```json
{
  "id": 0,
  "start": 0.0,
  "end": 3.5,
  "text": "Hello world"
}
```

This output feeds into the next pipeline stage (translation via argostranslate).